### **Walmart Sales Forecasting - Data Cleaning**

### **1. Import Libraries**

In [20]:
import pandas as pd
from pathlib import Path
import sys
from src.data import load_full_data, clean_data, validate_clean_data

### **2. Set Project Path**

In [21]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

### **3. Missing Values Before Cleaning**

In [22]:
merged_data = load_full_data()
display(merged_data.head())

INFO:root:Loading raw data...


INFO:root:Train shape: (421570, 5)
INFO:root:Test shape: (115064, 4)
INFO:root:Combined train/test shape: (536634, 6)
INFO:root:Final merged dataset shape: (536634, 17)


,Store,Dept,Date,Weekly_Sales,IsHoliday,is_train,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.5,False,1,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,1,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,1,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,1,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.9,False,1,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [23]:
merged_data["is_train"].value_counts()

is_train
1    421570
0    115064
Name: count, dtype: int64

In [24]:
missing_before = merged_data.isnull().sum().to_frame("missing_before")
display(missing_before[missing_before["missing_before"] > 0])

,missing_before
Weekly_Sales,115064
MarkDown1,271038
MarkDown2,338949
MarkDown3,294308
MarkDown4,299491
MarkDown5,270138
CPI,38162
Unemployment,38162


### **4. Clean Data**

In [25]:
cleaned_data = clean_data(merged_data)

print("Cleaned data shape:", cleaned_data.shape)
display(cleaned_data.head())

Cleaned data shape: (536634, 17)


,Store,Dept,Date,Weekly_Sales,IsHoliday,is_train,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,1,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,1,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,1,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,1,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,1,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,A,151315


In [26]:
cleaned_data["is_train"].value_counts()

is_train
1    421570
0    115064
Name: count, dtype: int64

In [27]:
cleaned_data[cleaned_data["is_train"] == 0]["Weekly_Sales"].isnull().sum()

np.int64(115064)

### **5. Validate Cleaned Data**

In [28]:
validate_clean_data(cleaned_data)

print("Duplicates after cleaning:", cleaned_data.duplicated().sum())
display(cleaned_data.isnull().sum().to_frame("missing_after"))

Duplicates after cleaning: 0


,missing_after
Store,0
Dept,0
Date,0
Weekly_Sales,115064
IsHoliday,0
is_train,0
Temperature,0
Fuel_Price,0
MarkDown1,0
MarkDown2,0


In [29]:
cleaned_data.groupby("is_train")["Weekly_Sales"].apply(lambda x: x.isnull().sum())

is_train
0    115064
1         0
Name: Weekly_Sales, dtype: int64

**Missing values in `Weekly_Sales` are expected for test rows because the test dataset represents future records where sales must be predicted. These rows are kept for final forecasting and will not be used during model training.**

### **6. Compare Before vs After**

In [30]:
summary = pd.DataFrame({
    "before": merged_data.isnull().sum(),
    "after": cleaned_data.isnull().sum()
})

display(summary[summary["before"] > 0])

,before,after
Weekly_Sales,115064,115064
MarkDown1,271038,0
MarkDown2,338949,0
MarkDown3,294308,0
MarkDown4,299491,0
MarkDown5,270138,0
CPI,38162,0
Unemployment,38162,0


**The cleaned dataset contains both historical training records and future test records. 
The `is_train` flag allows us to separate them later:**
- `is_train = 1` → training/validation data
- `is_train = 0` → final Kaggle forecasting data

The test rows correctly have missing `Weekly_Sales` because those are the values the model must predict.

### **7. Save Clean Dataset**

In [31]:
output_path = DATA_PROCESSED / "walmart_clean.csv"
cleaned_data.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: c:\Users\HP\Desktop\Practice\Projects\Walmart Sales Forecasting\data\processed\walmart_clean.csv


### **8. Cleaning Decisions**

- The merged dataset was loaded using the reusable `load_full_data()` function from `src/data/loader.py`.
- Date columns were converted to datetime format for time-series analysis.
- Missing markdown values were filled with 0 because missing markdowns likely represent no active promotion.
- Economic and external variables such as CPI, unemployment, temperature, and fuel price were forward-filled because they are slow-changing time-based features.
- Duplicate rows were removed to avoid double-counting sales records.
- Data was sorted by Store, Department, and Date to preserve time-series order.
- The cleaned dataset was saved as `data/processed/walmart_clean.csv`.